# 10 — Combined Part B + Part D dataset (2019–2023)

Not in the reference paper (it models each program separately). One row per NPI + year that appears in **either** program.

- **Outer join** of Part B AE (Notebook 03/04) and Part D AE (Notebook 09) on NPI + year.
- Feature columns prefixed `b_` (86) and `d_` (80); provider types kept as `b_type`, `d_type`; flags `has_partb`, `has_partd`.
- A provider missing from one program gets **NaN** for that program's features (not 0: "did not bill" ≠ "billed zero"). XGBoost handles NaN natively.
- Labels are recomputed from the cumulative LEIE table (Notebook 04) with the same rule as both programs, and checked against each program's own label.

Label columns (`-1` = row excluded from training and testing):
- `fraud` — paper rule: `year < last_excl_year`.
- `fraud_h2` — positives only if the exclusion is ≤ 2 years after the claim year; other `fraud = 1` rows are **dropped** (−1), not relabeled 0. (Fixes the Notebook 07 version, which relabeled them 0.)
- `fraud_nob4` — positives from 1128(b)(4) license revocations dropped (−1); tests whether license cases dilute the billing signal.

Output: `processed/combined/bd_labeled_2019_2023.parquet`

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
B_LAB    = DATA_DIR / "processed/part_b/labeled/part_b_labeled_2019_2023.parquet"
D_LAB    = DATA_DIR / "processed/part_d/labeled/part_d_AE_labeled_2019_2023.parquet"
LEIE_CUM = DATA_DIR / "processed/part_b/labeled/leie_cumulative_2019_2023.parquet"
OUT_DIR  = DATA_DIR / "processed/combined"; OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT      = OUT_DIR / "bd_labeled_2019_2023.parquet"
HORIZON  = 2
for p in (B_LAB, D_LAB, LEIE_CUM): print(p.exists(), p)

True /scratch/sm3664/medicare-fraud/processed/part_b/labeled/part_b_labeled_2019_2023.parquet
True /scratch/sm3664/medicare-fraud/processed/part_d/labeled/part_d_AE_labeled_2019_2023.parquet
True /scratch/sm3664/medicare-fraud/processed/part_b/labeled/leie_cumulative_2019_2023.parquet


## Step 1 — Load and prefix each program

In [2]:
b = pd.read_parquet(B_LAB)
b = b.rename(columns={"Rndrng_NPI": "npi", "fraud": "chk_b_fraud",
                      "Rndrng_Prvdr_Type": "b_type", "Rndrng_Prvdr_Ent_Cd": "b_ent_cd"})
b["b_type"] = b["b_type"].astype(str); b["b_ent_cd"] = b["b_ent_cd"].astype(str)
B_FEATS = [c for c in b.columns if c not in ("npi", "year", "chk_b_fraud", "b_type", "b_ent_cd")]
assert len(B_FEATS) == 86, len(B_FEATS)
b = b.rename(columns={c: "b_" + c for c in B_FEATS})
b[["b_" + c for c in B_FEATS]] = b[["b_" + c for c in B_FEATS]].astype("float32")

d = pd.read_parquet(D_LAB).drop(columns=["fraud_h2", "fraud_h3"])
d = d.rename(columns={"fraud": "chk_d_fraud", "provider_type": "d_type"})
D_FEATS = [c for c in d.columns if c not in ("npi", "year", "chk_d_fraud", "d_type")]
assert len(D_FEATS) == 80, len(D_FEATS)
d = d.rename(columns={c: "d_" + c for c in D_FEATS})
d[["d_" + c for c in D_FEATS]] = d[["d_" + c for c in D_FEATS]].astype("float32")
b, d = b.copy(), d.copy()          # defragment before the join
print("Part B", b.shape, "| Part D", d.shape)

Part B (5626423, 91) | Part D (5148303, 84)


## Step 2 — Outer join

In [3]:
bd = b.merge(d, on=["npi", "year"], how="outer", validate="one_to_one", indicator=True)
del b, d
bd["has_partb"] = bd["_merge"].isin(["both", "left_only"]).astype("int8")
bd["has_partd"] = bd["_merge"].isin(["both", "right_only"]).astype("int8")
bd["group"] = bd["_merge"].map({"both": "both", "left_only": "B_only", "right_only": "D_only"}).astype(str)
bd = bd.drop(columns="_merge").copy()
print(bd.shape)
print(pd.crosstab(bd.year, bd.group, margins=True))

(7425062, 176)


group   B_only   D_only     both      All
year                                     
2019    438410   330575   654957  1423942
2020    443305   341627   642008  1426940
2021    461172   354995   662417  1478584
2022    465205   373896   683668  1522769
2023    468667   397546   706614  1572827
All    2276759  1798639  3349664  7425062


## Step 3 — Labels from the cumulative LEIE table

In [4]:
cum = pd.read_parquet(LEIE_CUM)[["NPI", "EXCLTYPE", "EXCLDATE", "last_excl_year"]]
assert cum.NPI.is_unique
n0 = len(bd)
bd = bd.merge(cum, how="left", left_on="npi", right_on="NPI").drop(columns="NPI")
assert len(bd) == n0

bd["fraud"] = (bd.year < bd.last_excl_year).fillna(False).astype("int8")      # same rule as Notebooks 04 / 07

# Must reproduce each program's own label exactly.
for prog in ("b", "d"):
    m = bd[f"chk_{prog}_fraud"].notna()
    diff = (bd.loc[m, "fraud"] != bd.loc[m, f"chk_{prog}_fraud"]).sum()
    print(f"Part {prog.upper()}: {m.sum():,} rows, label mismatches: {diff}")
    assert diff == 0, f"Part {prog.upper()} label differs — check LABEL_RULE / LEIE file"
bd = bd.drop(columns=["chk_b_fraud", "chk_d_fraud"])

pos = bd.fraud == 1
far = (bd.EXCLDATE.dt.year - bd.year) > HORIZON
bd["fraud_h2"]   = bd.fraud.where(~(pos & far), -1).astype("int8")
bd["fraud_nob4"] = bd.fraud.where(~(pos & (bd.EXCLTYPE == "1128b4")), -1).astype("int8")
LABELS = ["fraud", "fraud_h2", "fraud_nob4"]

Part B: 5,626,423 rows, label mismatches: 0
Part D: 5,148,303 rows, label mismatches: 0


## Step 4 — Summary

In [5]:
for lab in LABELS:
    used = bd[bd[lab] >= 0]
    t = used.groupby("group").agg(rows=(lab, "size"), fraud_rows=(lab, "sum"),
                                  fraud_npis=("npi", lambda s: s[used.loc[s.index, lab] == 1].nunique()))
    t.loc["all"] = [len(used), int(used[lab].sum()), used.loc[used[lab] == 1, "npi"].nunique()]
    t["rate_%"] = (100 * t.fraud_rows / t.rows).round(4)
    print(f"\n{lab}  (dropped rows: {(bd[lab] == -1).sum():,})\n{t}")
    y = used.groupby("year")[lab].mean().mul(100).round(4)
    print("rate % by year:", y.to_dict())


fraud  (dropped rows: 0)
           rows  fraud_rows  fraud_npis  rate_%
group                                          
B_only  2276759         684         299  0.0300
D_only  1798639        1306         651  0.0726
both    3349664        1968         694  0.0588
all     7425062        3958        1291  0.0533
rate % by year: {2019: 0.0836, 2020: 0.0681, 2021: 0.0538, 2022: 0.0388, 2023: 0.0261}



fraud_h2  (dropped rows: 2,863)
           rows  fraud_rows  fraud_npis  rate_%
group                                          
B_only  2276212         137         111  0.0060
D_only  1797872         539         402  0.0300
both    3348115         419         298  0.0125
all     7422199        1095         709  0.0148
rate % by year: {2019: 0.0126, 2020: 0.0147, 2021: 0.0145, 2022: 0.0164, 2023: 0.0155}



fraud_nob4  (dropped rows: 1,799)
           rows  fraud_rows  fraud_npis  rate_%
group                                          
B_only  2276464         389         162  0.0171
D_only  1798083         750         401  0.0417
both    3348716        1020         379  0.0305
all     7423263        2159         748  0.0291
rate % by year: {2019: 0.0497, 2020: 0.0383, 2021: 0.0286, 2022: 0.0194, 2023: 0.012}


In [6]:
bd = bd.drop(columns=["EXCLTYPE", "EXCLDATE", "last_excl_year"])
bd.to_parquet(OUT, index=False)
print(OUT, bd.shape)

/scratch/sm3664/medicare-fraud/processed/combined/bd_labeled_2019_2023.parquet (7425062, 177)
